# 2 AG News (topic, 4 classes)

## Setup and Import Libraries

In [4]:
import re
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Load the dataset

In [5]:
url = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv"
df = pd.read_csv(url, header=None, names=["label", "title", "desc"])

# Use a subset for faster training (remove this line to use all 120k rows)
df = df.sample(n=30000, random_state=SEED).reset_index(drop=True)

df["text"] = (df["title"] + " " + df["desc"]).astype(str)
df["label"] = df["label"] - 1  # shift 1-4 -> 0-3

print(df[["label", "text"]].head())
print("Shape:", df.shape)
print("Class distribution:\n", df["label"].value_counts())

   label                                               text
0      2  BBC set for major shake-up, claims newspaper L...
1      2  Marsh averts cash crunch Embattled insurance b...
2      1  Jeter, Yankees Look to Take Control (AP) AP - ...
3      3  Flying the Sun to Safety When the Genesis caps...
4      2  Stocks Seen Flat as Nortel and Oil Weigh  NEW ...
Shape: (30000, 4)
Class distribution:
 label
1    7560
3    7528
0    7472
2    7440
Name: count, dtype: int64


## Clean and tokenize

In [6]:
def clean_text(s):
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text"] = df["text"].apply(clean_text)
df["tokens"] = df["text"].apply(str.split)
df[["text", "tokens"]].head()

,text,tokens
0,bbc set for major shake up claims newspaper lo...,"[bbc, set, for, major, shake, up, claims, news..."
1,marsh averts cash crunch embattled insurance b...,"[marsh, averts, cash, crunch, embattled, insur..."
2,jeter yankees look to take control ap ap derek...,"[jeter, yankees, look, to, take, control, ap, ..."
3,flying the sun to safety when the genesis caps...,"[flying, the, sun, to, safety, when, the, gene..."
4,stocks seen flat as nortel and oil weigh new y...,"[stocks, seen, flat, as, nortel, and, oil, wei..."


## Build Vocab